## "Orders at Risk" Analysis

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("../data/processed/product_volume_and_delay_dataset.csv")
pd.set_option('display.max_columns', None)

In [ ]:
# Group by order_id to get unique orders and their values
order_agg = df.groupby('order_id').agg({
    'product_weight_g': 'sum',
    'product_volume_cm3': 'sum',
    'freight_value': 'sum',
    'is_delayed': 'first'
}).reset_index()

# Risk Score calculator
def calculate_risk_score(row):
    score = 0
    if row["product_weight_g"] > 800:
        score += 1
    if row["product_volume_cm3"] > 7134:
        score += 1
    if row["freight_value"] > 20.00:
        score += 1
    return score

order_agg["delay_risk_score"] = order_agg.apply(calculate_risk_score, axis=1)

In [ ]:
delay_risk_metrics = order_agg.groupby("delay_risk_score")
# Failure Rate Calculator
def calculate_failure_rate(group):
    total_orders = group["order_id"].nunique()
    delayed_orders = group[group["is_delayed"] == True]["order_id"].nunique()
    failure_rate = round((delayed_orders / total_orders) * 100, 2)
    return f"{failure_rate}%"
    
final_table = delay_risk_metrics.apply(calculate_failure_rate).reset_index()
final_table.rename(columns={0: "failure_rate"})

### Observation

The simple risk score (0-3 points based on weight, volume, and freight value) shows that orders with the highest risk score of 3 have the worst failure rate at 7.76%, demonstrating that even a basic weighted scoring system can help identify shipments needing proactive intervention.

In [ ]:
# df.to_csv("../data/processed/at_risk_orders.csv", index=False)